intro_to_Language_modeling part1

[link](https://youtu.be/TCH_1BHY58I)

Previous model was for predicting next char when a single char was given and we had only one layer.
Now we will improve the model to handle multiple input char(3 in this case) with multiple hidden layer

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# download the training data
!curl https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt -o names.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  222k  100  222k    0     0   519k      0 --:--:-- --:--:-- --:--:--  519k


In [3]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# build the vocabulary of characters and lookup mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [21]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:

  print(f'{w=}')
  context = [0] * block_size # create a list with len 3
  for ch in w + '.':
    ix = stoi[ch] # get char's index
    X.append(context) # initially it will be [0,0,0]
    Y.append(ix)
    #print(context, '--->', itos[ix])
    print('given '+''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append. eg [0,0] + [5] = [0,0,5] . where 5 is the index of char 'e'

X = torch.tensor(X)
Y = torch.tensor(Y)

w='emma'
given ... ---> e
given ..e ---> m
given .em ---> m
given emm ---> a
given mma ---> .
w='olivia'
given ... ---> o
given ..o ---> l
given .ol ---> i
given oli ---> v
given liv ---> i
given ivi ---> a
given via ---> .
w='ava'
given ... ---> a
given ..a ---> v
given .av ---> a
given ava ---> .
w='isabella'
given ... ---> i
given ..i ---> s
given .is ---> a
given isa ---> b
given sab ---> e
given abe ---> l
given bel ---> l
given ell ---> a
given lla ---> .
w='sophia'
given ... ---> s
given ..s ---> o
given .so ---> p
given sop ---> h
given oph ---> i
given phi ---> a
given hia ---> .


In [22]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)